<a href="https://colab.research.google.com/github/Priti-Kannaujiya/DeepLearning-FromScratch/blob/main/SentimentAnalysis(CNN_LSTM).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
9import kagglehub
path = kagglehub.dataset_download("ashirwadsangwan/imdb-dataset")

100%|██████████| 1.64G/1.64G [00:12<00:00, 143MB/s]

Extracting files...


In [ ]:
import os
os.listdir(path)

['name.basics.tsv',
 'title.basics.tsv',
 'title.principals.tsv',
 'title.ratings.tsv',
 'title.akas.tsv']

In [ ]:
import pandas as pd
basics = pd.read_csv(path + "/title.basics.tsv", sep="\t")
basics.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short


In [ ]:
rating = pd.read_csv(path + "/title.ratings.tsv", sep='\t')
rating.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2198
1,tt0000002,5.5,310
2,tt0000003,6.5,2304
3,tt0000004,5.1,196
4,tt0000005,6.2,3035


In [ ]:
df=basics.merge(rating,on='tconst')

In [ ]:
df.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short",5.7,2198
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short",5.5,310
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance",6.5,2304
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short",5.1,196
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short,6.2,3035


In [ ]:
df.isnull().sum()

,0
tconst,0
titleType,0
primaryTitle,2
originalTitle,2
isAdult,0
startYear,0
endYear,0
runtimeMinutes,0
genres,4
averageRating,0


In [ ]:
df.shape

(1646044, 11)

In [ ]:
df=df[['primaryTitle','genres','averageRating']]
df=df.dropna()

In [ ]:
df.shape

(1646038, 3)

In [ ]:
df['label'] = df['averageRating'].apply(lambda x: 1 if x >= 7 else 0)

In [ ]:
df.head()

,primaryTitle,genres,averageRating,label
0,Carmencita,"Documentary,Short",5.7,0
1,Le clown et ses chiens,"Animation,Short",5.5,0
2,Poor Pierrot,"Animation,Comedy,Romance",6.5,0
3,Un bon bock,"Animation,Short",5.1,0
4,Blacksmith Scene,Short,6.2,0


In [ ]:
df['label'].value_counts()

,count
label,
1,937688
0,708350


In [ ]:
len(df['genres'].unique())

2072

In [ ]:
df = df.explode('genres')
df = pd.get_dummies(df, columns=['genres'])

In [ ]:
df.head()

,primaryTitle,averageRating,label,genres_Action,"genres_Action,Adult","genres_Action,Adult,Adventure","genres_Action,Adult,Animation","genres_Action,Adult,Comedy","genres_Action,Adult,Crime","genres_Action,Adult,Drama",...,"genres_Sport,Talk-Show","genres_Sport,Thriller","genres_Sport,War",genres_Talk-Show,genres_Thriller,"genres_Thriller,War","genres_Thriller,Western",genres_War,genres_Western,genres_\N
0,Carmencita,5.7,0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,Le clown et ses chiens,5.5,0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,Poor Pierrot,6.5,0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,Un bon bock,5.1,0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,Blacksmith Scene,6.2,0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
df.shape

(1646038, 2075)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
texts = df['primaryTitle']
tokenizer = Tokenizer(num_words=10000)       # we are taking top 10000 words that are appearing frequently
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

In [ ]:
sequences

[[],
 [63, 1290, 269, 7399],
 [1494],
 [107, 2481],
 [1308],
 [1705, 368],
 [6, 7239, 315, 1],
 [1629, 3, 5, 9349],
 [354, 1466],
 [1917, 1, 1322],
 [],
 [1, 2196, 3, 5, 455],
 [1, 8534, 5339, 7],
 [1],
 [606, 5, 1821],
 [1289, 1917, 1, 2869],
 [],
 [149],
 [1, 1290, 4067],
 [1, 3214],
 [1308],
 [1, 192],
 [1618, 3, 1, 4368],
 [1, 9486, 6, 3900, 1289, 362],
 [1, 8869, 28, 2621],
 [1712, 7],
 [1898, 14, 6771],
 [7555, 4252],
 [1994, 192, 28, 9636],
 [4686, 1, 6088],
 [43],
 [603, 1286, 2053],
 [2377, 455, 13],
 [1, 1183],
 [1342, 3, 4088],
 [192],
 [1, 457, 74],
 [603, 848],
 [175, 2869, 13],
 [13],
 [1437, 18, 6089],
 [347],
 [63],
 [118],
 [9113, 13],
 [6509, 177],
 [1, 3200, 6041],
 [3200, 613, 71, 7320, 1858],
 [269],
 [1],
 [670],
 [5, 69, 347, 1948, 69, 8216],
 [5, 1204, 89, 471],
 [63],
 [1657, 13, 12, 9487],
 [13, 347, 9793],
 [13, 670, 9113, 13],
 [6260, 3, 5, 923, 2925],
 [784],
 [784, 122],
 [5543],
 [938],
 [],
 [7917, 2171],
 [556],
 [145, 648, 5258, 7, 5136, 1879],
 [1, 12

In [ ]:
max_len = max([len(x) for x in sequences])
max_len

48

In [ ]:
sequences=pad_sequences(sequences,maxlen=max_len,padding='post')

In [ ]:
sequences[:5]

array([[   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0],
       [  63, 1290,  269, 7399,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0],
       [1494,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0],
       [ 107,

In [ ]:
len(tokenizer.word_index)

375438

In [ ]:
y=df['label']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    sequences, y, test_size=0.2, random_state=42
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential()
model.add(Embedding(input_dim=10000, output_dim=128, input_shape=(48,)))
model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(LSTM(64))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 48, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 46, 64)         │        24,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 23, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,341,889 (5.12 MB)

 Trainable params: 1,341,889 (5.12 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
history=model.fit(X_train,y_train,epochs=10,validation_data=(X_test,y_test))

Epoch 1/10
41151/41151 ━━━━━━━━━━━━━━━━━━━━ 368s 9ms/step - accuracy: 0.5759 - loss: 0.6730 - val_accuracy: 0.5917 - val_loss: 0.6620
Epoch 2/10
41151/41151 ━━━━━━━━━━━━━━━━━━━━ 392s 10ms/step - accuracy: 0.6013 - loss: 0.6560 - val_accuracy: 0.5946 - val_loss: 0.6607
Epoch 3/10
41151/41151 ━━━━━━━━━━━━━━━━━━━━ 386s 9ms/step - accuracy: 0.6117 - loss: 0.6477 - val_accuracy: 0.5936 - val_loss: 0.6602
Epoch 4/10
41151/41151 ━━━━━━━━━━━━━━━━━━━━ 433s 9ms/step - accuracy: 0.6204 - loss: 0.6399 - val_accuracy: 0.5940 - val_loss: 0.6624
Epoch 5/10
41151/41151 ━━━━━━━━━━━━━━━━━━━━ 370s 9ms/step - accuracy: 0.6276 - loss: 0.6324 - val_accuracy: 0.5900 - val_loss: 0.6646
Epoch 6/10
41151/41151 ━━━━━━━━━━━━━━━━━━━━ 356s 9ms/step - accuracy: 0.6353 - loss: 0.6247 - val_accuracy: 0.5915 - val_loss: 0.6720
Epoch 7/10
41151/41151 ━━━━━━━━━━━━━━━━━━━━ 349s 8ms/step - accuracy: 0.6415 - loss: 0.6180 - val_accuracy: 0.5903 - val_loss: 0.6735
Epoch 8/10
 6385/41151 ━━━━━━━━━━━━━━━━━━━━ 4:21 8ms/step - a